# CrisisQ: non-clinical claim verification and controlled revision experiments

**Compatibility update:** handles Transformers releases that replaced `warmup_ratio` with fractional `warmup_steps`. Uses a new `crisisq_nonclinical_v1_compat` output directory so existing frozen records are preserved. Restart the kernel and Run All after importing this update.

This notebook tests the submission's empirical claims as **English text classification and compression claims**. It does not require clinicians, infer clinical risk, or certify safety. Author-written rules are linguistic heuristics. Public dataset labels remain dataset/proxy labels. The reviewers may still require stronger validation or narrower claims; removing clinical terminology does not resolve evaluation leakage or labeling uncertainty.

**Inputs:** upload `crisisq_revision_results.zip` from your main notebook to Kaggle Add Input (extracted directories also work). GPU + internet are needed for new training. No gated dataset download is needed: existing exported splits are reused. Old final splits are explicitly **previously observed replication data**.

**Run in two stages:** Run All with `RUN_FINAL=False` to train and select using development data only. Review the development status and download the frozen plan. Then set `RUN_FINAL=True` and Run All again. Cached work is reused; final inference is opened only after the plan is immutable. Do not change rules, budgets, model recipes, data or selection after opening final results. A hash records consistency, not historical independence.

Optional: add `fresh_test.csv`, `fresh_probe.csv`, `fresh_hn.csv` with `text,label,group_id` (binary labels), authored/sourced independently after previous experiments. Add these before the first protocol freeze; adding data later requires a separately documented output directory/protocol. Add source and profile columns if available. These are optional, and their independence cannot be verified by code. Use a new independent collection after any final-driven redesign. For non-clinical annotation use two independent readers and a documented text-labeling guideline; ambiguous disclosures should stay uncertain, not be forced negative.

**What is new here:** fixed-shape CPU inference for both FP32 and INT8; mandatory PyTorch/ONNX parity check; end-to-end hybrid acceptance as well as neural acceptance; rejection without a least-violating fallback; a single variant selected on development data per arm/seed; repeated-seed inference on seed differences rather than pretending seed×example pairs are independent; audit-only original seed-42/0.63 configuration; independent external data is optional and never invented.

New recipes and backend searches are post-hoc revision experiments. A new passing configuration cannot retroactively validate the original one. Zero additional flips does not imply high recall. All pass/fail budgets below are engineering assumptions, not clinically justified cutoffs.


**Packaging fix:** the final cell now builds a small reports/cache ZIP and keeps model weights in place. The experimental cells and their existing frozen fingerprint are preserved. This packaging-only patch is recorded inside the reports ZIP. If your previous Kaggle session is still available, run only the recovery script first; do not retrain.


In [ ]:
# Install only missing dependencies. Restart the kernel if package installation requests it.
import importlib.util, subprocess, sys
pkgs={'numpy':'numpy','pandas':'pandas','scipy':'scipy','sklearn':'scikit-learn',
      'torch':'torch','transformers':'transformers','onnx':'onnx','onnxruntime':'onnxruntime',
      'accelerate':'accelerate','matplotlib':'matplotlib'}
missing=[pkg for mod,pkg in pkgs.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q']+missing)


In [ ]:
from pathlib import Path
import os, json, hashlib, random, gc, time, shutil, zipfile, platform, inspect, re
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import onnx, onnxruntime as ort
from scipy.stats import beta, t as student_t
from sklearn.metrics import roc_auc_score, average_precision_score, cohen_kappa_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed
from onnxruntime.quantization import quantize_dynamic, QuantType

INPUT_OVERRIDE = None     # path to extracted main results folder, or its ZIP
FRESH_DIR = None          # optional directory containing fresh_*.csv
RUN_FINAL = False         # first run False; second run True without any other changes
SMOKE_TEST = False        # True uses tiny training; never report its results
# Pair CE and SAQS over ten seeds. Factorial loss/augmentation ablations use three.
SEEDS = [42,123,777,1,2,3,4,5,6,7]
FACTORIAL_SEEDS = [42,123,777]
VARIANTS = ['qint8','q_perchannel_rr','q_quint8']
# Fixed padding and batch=1 reflect individual-message inference and avoid batch composition confounds.
INFERENCE_BATCH = 1
MAX_LENGTH=128; EPOCHS=2; BATCH=8; LR=2e-5
DEFAULT_TAU=0.70
# Original acceptance budgets, plus a separately reported absolute-recall guard.
BUDGETS=dict(hpf_probe=0.0,hpf_cal=0.01,recall_drop=0.02,
             cal_fpr_increase=0.10,probe_fpr_increase=0.05,hn_alert_rate=0.15)
ABS_MIN_CAL_RECALL=0.85  # engineering guard; NOT clinical adequacy
N_BOOT=2000
ROOT=Path('/kaggle/working/crisisq_nonclinical_v1_compat') if Path('/kaggle/working').exists() else Path('./crisisq_nonclinical_v1_compat')
if SMOKE_TEST:
    SEEDS=[42]; FACTORIAL_SEEDS=[42]; EPOCHS=1; N_BOOT=100
ROOT=ROOT.with_name(ROOT.name+('_SMOKE' if SMOKE_TEST else ''))
ROOT.mkdir(parents=True,exist_ok=True)
for n in ['models','cache','tables','figures','mobile','data']: (ROOT/n).mkdir(exist_ok=True)
ARMS={'ce':dict(loss='ce',aug=False),'saqs':dict(loss='saqs',aug=True),
      'ce_aug':dict(loss='ce',aug=True),'saqs_loss':dict(loss='saqs',aug=False)}
def seeds_for(arm): return SEEDS if arm in ['ce','saqs'] else FACTORIAL_SEEDS

def canonical_hash(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str).encode()).hexdigest()
def text_hash(t): return hashlib.sha256(re.sub(r'\s+',' ',str(t)).strip().lower().encode()).hexdigest()
def atomic_json(path,obj):
    tmp=path.with_suffix('.tmp'); tmp.write_text(json.dumps(obj,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def freeze(path,obj):
    if path.exists():
        assert json.loads(path.read_text())==json.loads(json.dumps(obj,default=str)), \
            f'{path.name} differs. Do not overwrite an existing experiment; preserve it and document a new protocol.'
    else: atomic_json(path,obj)
    return canonical_hash(obj)
VERSIONS={k:importlib.import_module(k).__version__ for k in ['numpy','pandas','scipy','sklearn','torch','transformers','onnx','onnxruntime']}
print('Output:',ROOT,'GPU:',torch.cuda.is_available(), 'CPUs:',os.cpu_count())
print(VERSIONS)


## 1. Load exported splits and record their evaluation history
No automatic relabeling and no automatic use of public models with unknown training overlap. The old safety probe is development data; the old final sets remain replication sets. The notebook cannot prove a supplied fresh set is historically unseen. Exact and lexical overlap audits are safeguards, not proof of independence.


In [ ]:
def find_input():
    if INPUT_OVERRIDE: candidates=[Path(INPUT_OVERRIDE)]
    else:
        bases=[Path('/kaggle/input'),Path('/kaggle/working'),Path('.')]
        candidates=[]
        for b in bases:
            if b.exists():
                candidates += [p.parent for p in b.rglob('manifest.json') if (p.parent/'data/train.csv').exists()]
                candidates += list(b.rglob('crisisq_revision_results*.zip'))
    if not candidates: raise FileNotFoundError('Upload crisisq_revision_results.zip or set INPUT_OVERRIDE.')
    p=sorted(set(candidates),key=str)[0]
    if p.suffix=='.zip':
        dest=ROOT/'original_input'; dest.mkdir(exist_ok=True)
        with zipfile.ZipFile(p) as z:
            for name in z.namelist():
                target=(dest/name).resolve()
                if dest.resolve() not in target.parents: raise ValueError('Unsafe archive path')
            z.extractall(dest)
        matches=[q.parent for q in dest.rglob('manifest.json') if (q.parent/'data/train.csv').exists()]
        if not matches: raise FileNotFoundError('ZIP does not contain main run data/train.csv')
        p=matches[0]
    return p
INPUT=find_input(); print('Input:',INPUT)
DEV=['cal','probe_dev','hn_dev']
OLD_FINAL=['test','probe_final','conflict','hn_final']
def read_frame(p):
    d=pd.read_csv(p,keep_default_na=False)
    assert {'text','label'} <= set(d.columns), f'Missing columns: {p}'
    d['text']=d.text.astype(str); d['label']=pd.to_numeric(d.label,errors='raise').astype(int)
    assert set(d.label)<= {0,1} and len(d)>0 and d.text.str.strip().ne('').all()
    d['audit_hash']=d.text.map(text_hash)
    assert not d.audit_hash.duplicated().any(), f'Duplicate text within {p}'
    if 'group_id' not in d:
        d['group_id']=d['group'].astype(str) if 'group' in d else d.audit_hash
    for col,default in [('profile','real_world'),('source','unspecified')]:
        if col not in d: d[col]=default
    return d.reset_index(drop=True)
DS={s:read_frame(INPUT/'data'/f'{s}.csv') for s in DEV+OLD_FINAL}
TRAIN=read_frame(INPUT/'data/train.csv'); AUG=read_frame(INPUT/'data/aug_clean.csv')
FRESH=[]
if FRESH_DIR:
    for s in ['fresh_test','fresh_probe','fresh_hn']:
        p=Path(FRESH_DIR)/f'{s}.csv'
        if p.exists():
            # Group IDs must be explicit for independent collection/template/conversation units.
            assert 'group_id' in pd.read_csv(p,nrows=0).columns, f'{p}: supply group_id'
            DS[s]=read_frame(p); FRESH.append(s)
FINAL=OLD_FINAL+FRESH
for s in ['cal','probe_dev','test','probe_final','conflict']:
    assert set(DS[s].label)=={0,1}, f'{s} must contain both classes'
for s in ['hn_dev','hn_final']+(['fresh_hn'] if 'fresh_hn' in FRESH else []):
    assert set(DS[s].label)=={0}, f'{s}: expected proposed negatives'
allframes={'train':TRAIN,'aug':AUG,**DS}
for i,a in enumerate(allframes):
    for b in list(allframes)[i+1:]:
        overlap=set(allframes[a].audit_hash)&set(allframes[b].audit_hash)
        assert not overlap, f'Exact text overlap {a}/{b}: {len(overlap)}. Resolve before training.'
assert not (set(DS['hn_dev'].group_id)&set(DS['hn_final'].group_id)), 'HN conversation-group overlap'
if SMOKE_TEST: TRAIN=TRAIN.groupby('label',group_keys=False).head(100).reset_index(drop=True)
print(pd.DataFrame([dict(split=s,n=len(d),positive=int(d.label.sum()),
    history='previously_observed' if s in OLD_FINAL else ('freshness_user_supplied' if s in FRESH else 'development')) for s,d in DS.items()]))
# Flag lexical neighbors for manual inspection. Do not silently delete final examples after evaluation.
from sklearn.feature_extraction.text import TfidfVectorizer
flags=[]
ref=pd.concat([TRAIN,AUG]+[DS[s] for s in DEV],ignore_index=True)
for s in FINAL:
    d=DS[s]; vec=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=1)
    z=vec.fit_transform(list(ref.text)+list(d.text)); rz=z[:len(ref)]
    for start in range(0,len(d),128):
        sim=(z[len(ref)+start:len(ref)+start+128]@rz.T).toarray()
        for j in np.flatnonzero(sim.max(axis=1)>=0.85):
            k=int(sim[j].argmax()); flags.append(dict(split=s,row=start+int(j),reference_row=k,similarity=float(sim[j,k]),
                text=d.text.iloc[start+j],reference=ref.text.iloc[k]))
pd.DataFrame(flags).to_csv(ROOT/'tables/lexical_overlap_flags.csv',index=False)
print('Lexical flags:',len(flags),'Review before starting; flagged overlap limits interpretation.')


## 2. Freeze the existing rules, recipe, versions, and splits
These are your previous v1/v2 linguistic rules, unchanged. The normalization algorithm and routing source are included in the fingerprint, not just the regex lists. No clinical framework mapping is asserted.


In [ ]:
def normalize_s1(t):
    t = str(t).lower()
    for ch in ["'", "’", "`", "‘"]:
        t = t.replace(ch, "")
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

# ---------------- v1 (legacy, as in the submitted manuscript) ----------------
V1_CRISIS = ["kill myself","end my life","take my own life","take my life","unalive myself","overdose","overdosing",
             "want to die","no reason to live","ending it all","end it all","better off dead","world would be better without me",
             "not waking up","hurt myself","harm myself","self harm","self-harm","hang myself","bought the pills","took too many",
             "slit my wrists","loaded the gun","want to end it","end my suffering","want to disappear permanently",
             "painless ways to go","writing my last","goodbye to everyone","said my goodbyes","take my own life","end it tonight"]
V1_SAFE = ["how can i support a suicidal","support a suicidal friend","help a suicidal friend","my sister is suicidal",
           "my brother is suicidal","my friend is suicidal","suicide prevention resources","researching suicide prevention",
           "research on suicide","i dont want to die","i do not want to die","i am not suicidal","im not suicidal","not suicidal",
           "dying of embarrassment","suicide prevention for a paper","not literally","prevention hotline",
           "how do i ask someone if they are suicidal","he said he wants to kill himself","she said she wants to end her life"]

def stage1_v1(text):
    t = normalize_s1(text)
    for p in V1_SAFE:
        if normalize_s1(p) in t:
            return 0, "safe_context"           # returns immediately: crisis cues are never checked
    for p in V1_CRISIS:
        if normalize_s1(p) in t:
            return 1, "vocab_crisis"
    return None, "uncertain"

# ---------------- v2 (revised) ----------------
# Each rule: (id, category, regex on NORMALISED text).  Categories are descriptive labels for expert
# review; any mapping to a clinical framework (e.g. C-SSRS) is a PROPOSAL until clinicians confirm it.
V2_CRISIS_RULES = [
    ("C01", "explicit_intent",      r"\bkill(?:ing)? myself\b"),
    ("C02", "explicit_intent",      r"\bend(?:ing)? my (?:own )?life\b"),
    ("C03", "explicit_intent",      r"\btak(?:e|ing) my (?:own )?life\b"),
    ("C04", "explicit_intent",      r"\b(?:end|ending) it all\b"),
    ("C05", "explicit_intent",      r"\bunalive myself\b"),
    ("C06", "explicit_wish_dead",   r"\bi (?:really |just |actually )?(?:want|wanna|need) to die\b"),
    ("C07", "explicit_wish_dead",   r"\bi wish (?:i was|i were|i am) dead\b|\bbetter off dead\b"),
    ("C08", "self_harm_intent",     r"\b(?:hurt|harm|hurting|harming) myself\b"),
    ("C09", "method_mention",       r"\bi (?:might |may |will |could |am going to |am thinking (?:of|about) |keep thinking about |thinking about )overdos(?:e|ing)\b"),
    ("C10", "method_mention",       r"\bhang(?:ing)? myself\b|\bslit my wrists\b|\bbought the pills\b|\btook too many (?:pills|tablets)\b"),
    ("C11", "preparation",          r"\bsaid my goodbyes\b|\bwriting my last (?:note|message|letter)\b|\bpainless ways? to go\b"),
    ("C12", "explicit_intent",      r"\bno reason to live\b|\bend my suffering\b|\bdisappear permanently\b"),
    ("C13", "explicit_intent",      r"\bi (?:am|m) going to (?:die|jump)\b"),
]
V2_SAFE_RULES = [
    ("S01", "third_party_support",  r"\b(?:support|help|helping|supporting) (?:a |my |an? )?(?:suicidal )?(?:friend|sister|brother|colleague|someone)\b.*\bsuicidal\b|\b(?:my|a) (?:friend|sister|brother|colleague)\b.{0,30}\bis suicidal\b|\bsomeone is suicidal\b"),
    ("S02", "third_party_support",  r"\bhow (?:can|do) i (?:support|help|ask) (?:a |an |my )?(?:suicidal|someone)"),
    ("S03", "prevention_research",  r"\bsuicide prevention\b"),
    ("S04", "prevention_research",  r"\bresearch(?:ing)? (?:on |about )?suicide\b"),
    ("S05", "explicit_negation",    r"\bi (?:do not|dont|am not|m not|am never) (?:want to die|suicidal)\b|\bnot suicidal\b|\bim not suicidal\b"),
    ("S06", "idiom",                r"\bdying of embarrassment\b|\bnot literally\b"),
    ("S07", "prevention_research",  r"\bprevention hotline\b|\bcrisis hotlines?\b"),
]
V2_NEGATORS = {"not","never","no","dont","wont","wouldnt","cant","cannot","without","didnt","isnt","arent","aint","nobody"}
V2_CONTRAST = re.compile(r"\b(?:but|however|although|though|yet|except|until|still)\b")
V2_RISK_CUES = re.compile(r"\b(?:tonight|pills?|plan|planning|goodbye|die|dies|dead|dying|kill|disappear|gone|hurt|harm|hopeless|worthless|burden|"
                          r"no point|never wake|not be alive|stay alive|staying alive|leave this world|give away|giving away|swallow|"
                          r"be here anymore|exist|existing|suicide|suicidal|end it|ending)\b")
_C = [(i, c, re.compile(p)) for i, c, p in V2_CRISIS_RULES]
_S = [(i, c, re.compile(p)) for i, c, p in V2_SAFE_RULES]

def _is_negated(t, start):
    prev = t[:start].split()[-4:]
    return any(w in V2_NEGATORS for w in prev)

def stage1_v2(text, return_trace=False):
    t = normalize_s1(text)
    aff, neg = [], []
    for rid, cat, rx in _C:
        for m in rx.finditer(t):
            (neg if _is_negated(t, m.start()) else aff).append(rid)
    safe_hits, spans = [], []
    for rid, cat, rx in _S:
        for m in rx.finditer(t):
            safe_hits.append(rid); spans.append(m.span())
    trace = dict(crisis_affirmed=sorted(set(aff)), crisis_negated=sorted(set(neg)), safe=sorted(set(safe_hits)))
    if aff and safe_hits:
        out = (None, "conflict_to_stage2")
    elif aff:
        out = (1, "crisis")
    elif neg:                                   # crisis words under negation, e.g. "I would never hurt myself"
        out = (None, "negated_crisis_to_stage2")
    elif safe_hits:
        resid = t
        for a, b in sorted(spans, reverse=True):
            resid = resid[:a] + " " + resid[b:]
        if V2_CONTRAST.search(t) or V2_RISK_CUES.search(resid):
            out = (None, "safe_with_residual_risk_to_stage2")
        else:
            out = (0, "safe_context")
    else:
        out = (None, "uncertain")
    return (out + (trace,)) if return_trace else out

def stage1_array(texts, version):
    fn = stage1_v1 if version == "v1" else stage1_v2
    return np.array([(-1 if fn(t)[0] is None else fn(t)[0]) for t in texts], dtype=int)

def rules_fingerprint():
    blob = json.dumps(dict(v2c=V2_CRISIS_RULES, v2s=V2_SAFE_RULES, neg=sorted(V2_NEGATORS),
                           contrast=V2_CONTRAST.pattern, risk=V2_RISK_CUES.pattern), sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]


In [ ]:
def frame_digest(d):
    return canonical_hash(d[['text','label','group_id','profile','source']].to_dict('records'))
# Pin the pretrained checkpoint; resumed runs reuse the frozen revision.
from huggingface_hub import model_info
prior_protocol=ROOT/'protocol_frozen.json'
BASE_REVISION=json.loads(prior_protocol.read_text())['base_revision'] if prior_protocol.exists() else model_info('distilbert-base-uncased').sha
GATE_SOURCE = inspect.getsource(stage1_v2)+inspect.getsource(_is_negated)+inspect.getsource(normalize_s1)
RULE_SPEC=dict(crisis=V2_CRISIS_RULES,safe=V2_SAFE_RULES,negators=sorted(V2_NEGATORS),
               contrast=V2_CONTRAST.pattern,risk=V2_RISK_CUES.pattern,source=GATE_SOURCE,
               v1_crisis=V1_CRISIS,v1_safe=V1_SAFE)
PROTOCOL=dict(version='nonclinical_v1_compat',implementation_hash='30577c225cc2aafed10568f16414603b2cdb910df5c54e247a56bbd182b28746',arms=ARMS,seeds=SEEDS,factorial_seeds=FACTORIAL_SEEDS,
    variants=VARIANTS,budgets=BUDGETS,min_cal_recall=ABS_MIN_CAL_RECALL,
    epochs=EPOCHS,batch=BATCH,lr=LR,max_length=MAX_LENGTH,inference_batch=INFERENCE_BATCH,
    default_tau=DEFAULT_TAU,smoke=SMOKE_TEST,versions=VERSIONS,rules=RULE_SPEC,
    splits={s:frame_digest(d) for s,d in {'train':TRAIN,'aug':AUG,**DS}.items()},
    final_history={s:('previously_observed' if s in OLD_FINAL else 'freshness_user_supplied') for s in FINAL},
    selection='hybrid constraints; choose dev HN alert rate, cal FPR, cal recall, file size, fixed variant order',
    model_id='distilbert-base-uncased',base_revision=BASE_REVISION,
    environment=dict(platform=platform.platform(),processor=platform.processor(),cpu_count=os.cpu_count(),
                     torch_gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                     ort_providers=ort.get_available_providers(),ort_build=ort.get_build_info()))
PROTOCOL_HASH=freeze(ROOT/'protocol_frozen.json',PROTOCOL)
GATES={v:{s:stage1_array(d.text,v) for s,d in DS.items()} for v in ['v1','v2']}
pd.DataFrame([dict(rule_id=i,family=f,category=c,pattern=p,provenance='author-written linguistic heuristic')
    for f,rules in [('crisis',V2_CRISIS_RULES),('safe_context',V2_SAFE_RULES)] for i,c,p in rules]).to_csv(ROOT/'tables/rule_inventory.csv',index=False)
print('Protocol frozen:',PROTOCOL_HASH)


## 3. Metrics, rejection-only threshold selection, and uncertainty
The original budget test is reported separately from the additional minimum calibration-recall guard. Each INT8 candidate is compared with its own FP32 model using identical routing. We evaluate neural and hybrid acceptance separately. Only a hybrid that passes **all** original budgets **and** the recall guard can be selected for the new system experiment. No feasible candidate means rejection, not a fallback deployment model.

Thresholds come from development scores only. A second comparison chooses thresholds to target 90% calibration recall for every model; test recall is allowed to differ. Any missed target is explicit.


In [ ]:
def predict(p,tau,gate=None):
    out=(np.asarray(p)>=tau).astype(int)
    return out if gate is None else np.where(gate<0,out,gate)
def metric(y,pred):
    y=np.asarray(y); pred=np.asarray(pred); pos=y==1; neg=y==0
    return dict(recall=float(pred[pos].mean()) if pos.any() else np.nan,
                fpr=float(pred[neg].mean()) if neg.any() else np.nan,
                missed=int((pos&(pred==0)).sum()),false_alarms=int((neg&(pred==1)).sum()),
                n_pos=int(pos.sum()),n_neg=int(neg.sum()))
def lost_rate(y,ref,q):
    y=np.asarray(y); pos=y==1
    return float(((ref==1)&(q==0)&pos).sum()/pos.sum()) if pos.any() else np.nan
def candidate_thresholds(p):
    # Include all change points and all-positive/all-negative boundaries. No coarse-grid saturation ceiling.
    v=np.unique(np.asarray(p,dtype=float))
    assert np.isfinite(v).all() and ((v>=0)&(v<=1)).all()
    return np.unique(np.r_[0.,v,np.nextafter(v,np.inf),np.nextafter(1.,np.inf)])
def select_threshold(y,p,ref,gate,budgets,min_recall=ABS_MIN_CAL_RECALL):
    # Inputs contain development splits ONLY. Returns tau=None for rejection.
    refs={s:predict(ref[s],DEFAULT_TAU,gate.get(s)) for s in DEV}
    mr={s:metric(y[s],refs[s]) for s in ['cal','probe_dev']}
    rows=[]
    # Decisions can change on any DEV split, so include every development score.
    for tau in candidate_thresholds(np.concatenate([p[s] for s in DEV])):
        q={s:predict(p[s],tau,gate.get(s)) for s in DEV}
        m={s:metric(y[s],q[s]) for s in DEV}
        checks=dict(hpf_probe=lost_rate(y['probe_dev'],refs['probe_dev'],q['probe_dev'])<=budgets['hpf_probe']+1e-12,
          hpf_cal=lost_rate(y['cal'],refs['cal'],q['cal'])<=budgets['hpf_cal']+1e-12,
          recall_drop=m['cal']['recall']>=mr['cal']['recall']-budgets['recall_drop']-1e-12,
          cal_fpr_increase=m['cal']['fpr']<=mr['cal']['fpr']+budgets['cal_fpr_increase']+1e-12,
          probe_recall=m['probe_dev']['recall']>=mr['probe_dev']['recall']-1e-12,
          probe_fpr_increase=m['probe_dev']['fpr']<=mr['probe_dev']['fpr']+budgets['probe_fpr_increase']+1e-12,
          hn_alert_rate=m['hn_dev']['fpr']<=budgets['hn_alert_rate']+1e-12)
        orig=all(checks.values()); guard=m['cal']['recall']>=min_recall-1e-12
        rows.append(dict(tau=float(tau),original_budgets_pass=orig,absolute_recall_guard=guard,
             accepted=orig and guard,cal_recall=m['cal']['recall'],cal_fpr=m['cal']['fpr'],
             hn_alert_rate=m['hn_dev']['fpr'],failed=','.join(k for k,v in checks.items() if not v)))
    feasible=[r for r in rows if r['accepted']]
    best=min(feasible,key=lambda r:(r['hn_alert_rate'],r['cal_fpr'],-r['cal_recall'],-r['tau'])) if feasible else None
    return dict(accepted=best is not None,tau=None if best is None else best['tau'],
                original_feasible=any(r['original_budgets_pass'] for r in rows),
                selected_dev=best),rows

def target_recall(y,p,gate,target=0.90):
    ok=[float(t) for t in candidate_thresholds(p) if metric(y,predict(p,t,gate))['recall']>=target-1e-12]
    return max(ok) if ok else None

def cp95(k,n):
    if n==0: return (np.nan,np.nan)
    return (0. if k==0 else float(beta.ppf(.025,k,n-k+1)),
            1. if k==n else float(beta.ppf(.975,k+1,n-k)))
def zero_upper(k,n):
    return 1. if k==n else float(beta.ppf(.95,k+1,n-k)) if n else np.nan

def seed_interval(d):
    d=np.asarray(d,float); n=len(d); mean=float(d.mean())
    if n<2: return mean,np.nan,np.nan
    half=float(student_t.ppf(.975,n-1)*d.std(ddof=1)/np.sqrt(n))
    return mean,mean-half,mean+half

def cluster_delta(y,a,b,groups,which='recall',B=N_BOOT):
    y=np.asarray(y); a=np.asarray(a); b=np.asarray(b); groups=np.asarray(groups)
    mask=(y==1) if which=='recall' else (y==0)
    g=np.unique(groups[mask]); diffs=[]; rng=np.random.default_rng(1729)
    if len(g)<2: return np.nan,np.nan
    # Resample entire independent groups, retaining all members and their repeated draws.
    for _ in range(B):
        idx=np.concatenate([np.flatnonzero((groups==v)&mask) for v in rng.choice(g,len(g),replace=True)])
        diffs.append(float((b[idx]-a[idx]).mean()))
    return tuple(np.quantile(diffs,[.025,.975]))


## 4. Train matched models, export, and verify numerical parity
All arms use the same backbone, training split, epochs and seed list. CE/SAQS use ten seeds; loss/augmentation factorial controls use three. SAQS reproduces the previous weighted CE plus margin loss. No new augmentation is generated. Training is GPU; all measured probabilities are ONNX CPU with fixed padding and batch size. Expect substantial CPU inference time; the cache resumes completed splits.


In [ ]:
from torch.utils.data import Dataset
from types import SimpleNamespace
POS_CLASS_WEIGHT=2.5; HARD_POS_WEIGHT=4.; HARD_NEG_WEIGHT=2.
POS_MARGIN=3.; NEG_MARGIN=3.; MARGIN_LAMBDA=.25
HARD_POS={'direct_crisis','indirect_crisis','passive_ideation'}
HARD_NEG={'third_party_support','informational','safe_context','idiom','distress_non_crisis','past_recovered','general_support'}
class CrisisDataset(Dataset):
    def __init__(self, df, tok): self.df, self.tok = df.reset_index(drop=True), tok
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        enc = self.tok(str(r["text"]), truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors=None)
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in enc.items()}
        lab = int(r["label"]); prof = str(r.get("profile", "real_world")); aug = str(r.get("source", "")) == "safety_aug_train"
        item["labels"]   = torch.tensor(lab, dtype=torch.long)
        item["hard_pos"] = torch.tensor(int(lab == 1 and (prof in HARD_POS or aug)), dtype=torch.float)
        item["hard_neg"] = torch.tensor(int(lab == 0 and (prof in HARD_NEG or aug)), dtype=torch.float)
        return item

class CrisisTrainer(Trainer):
    """loss_mode='ce': plain cross-entropy.  loss_mode='saqs': paper loss (class weight 2.5, hard-example weights, margin)."""
    def __init__(self, *a, loss_mode="saqs", **k):
        super().__init__(*a, **k); self.loss_mode = loss_mode
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels"); hp = inputs.pop("hard_pos", None); hn = inputs.pop("hard_neg", None)
        logits = model(**inputs).logits
        if self.loss_mode == "ce":
            loss = F.cross_entropy(logits, labels)
        else:
            ce = F.cross_entropy(logits, labels, reduction="none"); w = torch.ones_like(ce)
            w = torch.where(labels == 1, w * POS_CLASS_WEIGHT, w)
            if hp is not None: w = w * (1 + hp.to(w.device) * (HARD_POS_WEIGHT - 1))
            if hn is not None: w = w * (1 + hn.to(w.device) * (HARD_NEG_WEIGHT - 1))
            d = logits[:, 1] - logits[:, 0]
            ml = (F.relu(POS_MARGIN - d) * (labels == 1).float() + F.relu(NEG_MARGIN + d) * (labels == 0).float()).mean()
            loss = (ce * w).mean() + MARGIN_LAMBDA * ml
        return (loss, SimpleNamespace(logits=logits)) if return_outputs else loss


def tag(arm,seed): return f'{arm}__{seed}'
def model_dir(arm,seed): return ROOT/'models'/tag(arm,seed)
def ppath(arm,seed,variant,split): return ROOT/'cache'/f'{tag(arm,seed)}__{variant}__{split}.npy'
def loadp(arm,seed,variant,split): return np.load(ppath(arm,seed,variant,split))

def compatible_training_arguments(kwargs, argument_class=TrainingArguments):
    kwargs=dict(kwargs)
    supported=inspect.signature(argument_class.__init__).parameters
    if 'warmup_ratio' in kwargs and 'warmup_ratio' not in supported:
        ratio=kwargs.pop('warmup_ratio')
        if 'warmup_steps' not in supported:
            raise RuntimeError('This Transformers version provides no supported warmup argument.')
        # Recent Transformers accepts a float <1 as a fraction of total steps.
        kwargs['warmup_steps']=float(ratio)
    unknown=set(kwargs)-set(supported)
    if unknown:
        raise RuntimeError(f'Unsupported TrainingArguments options: {sorted(unknown)}. '
                           'Do not silently discard training settings.')
    return argument_class(**kwargs)

def train_export(arm,seed):
    dest=model_dir(arm,seed); dest.mkdir(exist_ok=True)
    if (dest/'export_complete.json').exists(): return dest
    set_seed(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tok=AutoTokenizer.from_pretrained('distilbert-base-uncased',revision=BASE_REVISION)
    model=AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased',revision=BASE_REVISION,num_labels=2)
    data=pd.concat([TRAIN,AUG],ignore_index=True) if ARMS[arm]['aug'] else TRAIN
    data=data.sample(frac=1,random_state=seed).reset_index(drop=True)
    kwargs=dict(output_dir=str(dest/'trainer'),num_train_epochs=EPOCHS,per_device_train_batch_size=BATCH,
        learning_rate=LR,weight_decay=.01,warmup_ratio=.06,save_strategy='no',report_to=[],
        seed=seed,fp16=torch.cuda.is_available(),remove_unused_columns=False,dataloader_num_workers=0)
    tr=CrisisTrainer(model=model,args=compatible_training_arguments(kwargs),train_dataset=CrisisDataset(data,tok),loss_mode=ARMS[arm]['loss'])
    # Custom per-batch loss; no gradient accumulation in this protocol.
    tr.model_accepts_loss_kwargs=False
    tr.train(); model=tr.model.module if hasattr(tr.model,'module') else tr.model
    model=model.float().cpu().eval(); tok.save_pretrained(dest); model.config.save_pretrained(dest)
    class Wrapper(torch.nn.Module):
        def __init__(self,m): super().__init__(); self.m=m
        def forward(self,input_ids,attention_mask): return self.m(input_ids=input_ids,attention_mask=attention_mask).logits
    example=tok('A neutral example for export.',padding='max_length',truncation=True,max_length=MAX_LENGTH,return_tensors='pt')
    w=Wrapper(model).eval()
    exportkw=dict(input_names=['input_ids','attention_mask'],output_names=['logits'],
        dynamic_axes={'input_ids':{0:'batch'},'attention_mask':{0:'batch'},'logits':{0:'batch'}},opset_version=17)
    if 'dynamo' in inspect.signature(torch.onnx.export).parameters: exportkw['dynamo']=False
    with torch.no_grad(): torch.onnx.export(w,(example['input_ids'],example['attention_mask']),str(dest/'fp32.onnx'),**exportkw)
    onnx.checker.check_model(onnx.load(str(dest/'fp32.onnx')))
    # Use development inputs only for export parity. Compare logits, not predicted labels alone.
    probe=tok(DS['cal'].text.iloc[:8].tolist(),padding='max_length',truncation=True,max_length=MAX_LENGTH,return_tensors='pt')
    with torch.no_grad(): expected=model(**probe).logits.numpy()
    session=ort.InferenceSession(str(dest/'fp32.onnx'),providers=['CPUExecutionProvider'])
    got=session.run(None,{k:v.numpy().astype(np.int64) for k,v in probe.items() if k in ['input_ids','attention_mask']})[0]
    err=float(np.max(np.abs(expected-got)))
    if not np.allclose(expected,got,atol=1e-3,rtol=1e-3): raise RuntimeError(f'Export parity failed: {err}')
    atomic_json(dest/'export_complete.json',dict(parity_max_abs_error=err,training_rows=len(data),protocol=PROTOCOL_HASH))
    del tr,model,w,session; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return dest

def quantize(dest,variant):
    out=dest/f'{variant}.onnx'
    if out.exists(): return out
    kw={'qint8':dict(weight_type=QuantType.QInt8),
        'q_perchannel_rr':dict(weight_type=QuantType.QInt8,per_channel=True,reduce_range=True),
        'q_quint8':dict(weight_type=QuantType.QUInt8)}[variant]
    tmp=dest/f'{variant}.partial.onnx'
    quantize_dynamic(str(dest/'fp32.onnx'),str(tmp),**kw)
    onnx.checker.check_model(onnx.load(str(tmp))); tmp.replace(out)
    return out

def infer(dest,variant,texts):
    so=ort.SessionOptions(); so.intra_op_num_threads=min(4,os.cpu_count() or 1)
    sess=ort.InferenceSession(str(dest/f'{variant}.onnx'),sess_options=so,providers=['CPUExecutionProvider'])
    tok=AutoTokenizer.from_pretrained(str(dest)); texts=list(texts); out=[]
    for i in range(0,len(texts),INFERENCE_BATCH):
        enc=tok(texts[i:i+INFERENCE_BATCH],padding='max_length',truncation=True,max_length=MAX_LENGTH,return_tensors='np')
        logits=sess.run(None,{k:enc[k].astype(np.int64) for k in ['input_ids','attention_mask']})[0].astype(np.float64)
        logits-=logits.max(axis=1,keepdims=True); e=np.exp(logits); out.extend((e/e.sum(axis=1,keepdims=True))[:,1])
    arr=np.asarray(out); assert len(arr)==len(texts) and np.isfinite(arr).all()
    return arr

FINAL_OPEN=False
def cache_split(arm,seed,variant,split):
    if split in FINAL and not FINAL_OPEN: raise RuntimeError('Final evaluation is locked')
    p=ppath(arm,seed,variant,split)
    if not p.exists():
        a=infer(model_dir(arm,seed),variant,DS[split].text)
        tmp=p.with_suffix('.tmp.npy'); np.save(tmp,a); tmp.replace(p)
    return loadp(arm,seed,variant,split)
for arm in ARMS:
    for seed in seeds_for(arm):
        print('Development:',arm,seed,flush=True)
        d=train_export(arm,seed)
        for variant in ['fp32']+VARIANTS:
            if variant!='fp32': quantize(d,variant)
            for split in DEV: cache_split(arm,seed,variant,split)
print('Development inference complete. Final sets have not been inferred.')


## 5. Select on development data and freeze the complete evaluation plan
Variant selection is per seed and uses the same predefined procedure in each arm. We report every candidate's development status, including rejected candidates. Comparing the hybrid with its neural counterpart uses the exact same selected weights and threshold. The separately selected neural-only threshold is also retained as a control for tuning effects.


In [ ]:
PLAN={}; DEV_ROWS=[]
ydev={s:DS[s].label.to_numpy() for s in DEV}
for arm in ARMS:
    for seed in seeds_for(arm):
        ref={s:loadp(arm,seed,'fp32',s) for s in DEV}; candidates=[]; allinfo={}
        for variant in VARIANTS:
            p={s:loadp(arm,seed,variant,s) for s in DEV}
            info={}
            for system in ['neural','hybrid']:
                gates={} if system=='neural' else {s:GATES['v2'][s] for s in DEV}
                selection,sweep=select_threshold(ydev,p,ref,gates,BUDGETS)
                info[system]=selection
                pd.DataFrame(sweep).to_csv(ROOT/'tables'/f'dev_sweep__{tag(arm,seed)}__{variant}__{system}.csv',index=False)
                DEV_ROWS.append(dict(arm=arm,seed=seed,variant=variant,system=system,**selection))
            allinfo[variant]=info
            if info['hybrid']['accepted']:
                b=info['hybrid']['selected_dev']; size=(model_dir(arm,seed)/f'{variant}.onnx').stat().st_size
                candidates.append((b['hn_alert_rate'],b['cal_fpr'],-b['cal_recall'],size,VARIANTS.index(variant),variant))
        winner=min(candidates)[-1] if candidates else None
        selected=None if winner is None else dict(variant=winner,tau=allinfo[winner]['hybrid']['tau'])
        # Fixed recall-target comparison: no variant search, default qint8 for everyone.
        matched={}
        for variant in ['fp32','qint8']:
            for system in ['neural','hybrid']:
                gate=None if system=='neural' else GATES['v2']['cal']
                matched[f'{variant}|{system}']=target_recall(ydev['cal'],loadp(arm,seed,variant,'cal'),gate)
        PLAN[tag(arm,seed)]=dict(arm=arm,seed=seed,selected=selected,candidates=allinfo,matched_R90=matched)
plan_obj=dict(protocol=PROTOCOL_HASH,models=PLAN)
SELECTION_HASH=freeze(ROOT/'selection_frozen.json',plan_obj)
devtab=pd.DataFrame(DEV_ROWS); devtab.to_csv(ROOT/'tables/development_acceptance.csv',index=False)
print(devtab[['arm','seed','variant','system','accepted','original_feasible','tau']].to_string(index=False))
print('Selected seeds by arm:',{arm:sum(v['selected'] is not None for v in PLAN.values() if v['arm']==arm) for arm in ARMS})
print('Selection hash:',SELECTION_HASH)
if not RUN_FINAL: print('STOP FOR REVIEW: final cells below are skipped. Set RUN_FINAL=True and rerun without other edits to evaluate.')


## 6. Final reporting: replication first, fresh data separately
Each original configuration, every fixed quantizer, and the development-selected system is evaluated for reporting only. No final metric feeds selection. Seed-42 INT8 at 0.63 is a configuration replication with retrained weights, not a rerun of the submission's original checkpoint. Those original weights are not supplied by the main results archive. Rejected candidates can be diagnosed, but never counted as accepted systems. Acceptance is attached to the hybrid; neural/v1 controls using its threshold have not independently earned that acceptance.

Hard-negative rows contain proposed negatives. Their reported `fpr` is a **provisional alert rate**, unless independently verified negative. Do not describe its reduction as a reduction in false alarms without label evidence. Fresh text groups enable cluster uncertainty; supply meaningful template/conversation IDs rather than treating paraphrases as independent examples.


In [ ]:
ROWS=[]; PREDS={}; FLIPS=[]; DELTAS=[]
def report(arm,seed,variant,split,name,tau,system,accepted=False):
    if tau is None: return
    p=loadp(arm,seed,variant,split); y=DS[split].label.to_numpy()
    gate=None if system=='neural' else GATES['v1' if system=='hybrid_v1' else 'v2'][split]
    pr=predict(p,tau,gate); m=metric(y,pr)
    rec_ci=cp95(m['n_pos']-m['missed'],m['n_pos']); fpr_ci=cp95(m['false_alarms'],m['n_neg'])
    key=(arm,seed,variant,split,name,system); PREDS[key]=pr
    ROWS.append(dict(arm=arm,seed=seed,variant=variant,split=split,analysis=name,system=system,tau=tau,
        hybrid_selection_passed=accepted,history=PROTOCOL['final_history'][split],
        negative_metric='provisional_alert_rate' if split.endswith('hn') or split=='hn_final' else 'dataset_label_FPR',
        size_mb=(model_dir(arm,seed)/f'{variant}.onnx').stat().st_size/1e6,
        stage2_rate=1. if gate is None else float((gate<0).mean()),
        recall_lo=rec_ci[0],recall_hi=rec_ci[1],fpr_lo=fpr_ci[0],fpr_hi=fpr_ci[1],
        ci_scope='binomial_descriptive; use group bootstrap for templated/dependent rows',**m))
    return pr
if RUN_FINAL:
    freeze(ROOT/'final_opened.json',dict(protocol=PROTOCOL_HASH,selection=SELECTION_HASH))
    FINAL_OPEN=True
    for item in PLAN.values():
        arm,seed=item['arm'],item['seed']; print('Final:',arm,seed,flush=True)
        for variant in ['fp32']+VARIANTS:
            for split in FINAL: cache_split(arm,seed,variant,split)
        for split in FINAL:
            y=DS[split].label.to_numpy()
            for variant in ['fp32']+VARIANTS:
                for system in ['neural','hybrid_v1','hybrid_v2']:
                    report(arm,seed,variant,split,'fixed_0.70',DEFAULT_TAU,system)
                for system in ['neural','hybrid']:
                    tau=item['matched_R90'].get(f'{variant}|{system}')
                    report(arm,seed,variant,split,'calibration_R90',tau,system)
            if arm=='saqs' and seed==42:
                for system in ['neural','hybrid_v1','hybrid_v2']:
                    report(arm,seed,'qint8',split,'seed42_tau0.63_retrained_config',.63,system)
            selected=item['selected']
            if selected:
                v,tau=selected['variant'],selected['tau']
                for system in ['neural','hybrid_v1','hybrid_v2']:
                    report(arm,seed,v,split,'dev_selected',tau,system,accepted=True)
                a=PREDS[(arm,seed,v,split,'dev_selected','neural')]
                b=PREDS[(arm,seed,v,split,'dev_selected','hybrid_v2')]
                for metric_name in ['recall','fpr']:
                    mask=(y==1) if metric_name=='recall' else (y==0)
                    if mask.any():
                        lo,hi=cluster_delta(y,a,b,DS[split].group_id.to_numpy(),metric_name)
                        DELTAS.append(dict(arm=arm,seed=seed,split=split,metric=metric_name,
                            hybrid_minus_neural=float((b[mask]-a[mask]).mean()),cluster_lo=lo,cluster_hi=hi))
            # Compression audit per candidate, same tau and repaired tau. Never pool exact CIs over seeds.
            for v in VARIANTS:
                options=[('fixed_tau',DEFAULT_TAU)]
                for sysname,sel in item['candidates'][v].items():
                    if sel['accepted']: options.append((f'dev_repaired_{sysname}',sel['tau']))
                for comparison,tau in options:
                    for system in ['neural','hybrid']:
                        gate=None if system=='neural' else GATES['v2'][split]
                        q=predict(loadp(arm,seed,v,split),tau,gate)
                        for refname,rtau in [('same_tau',tau),('fp32_default',DEFAULT_TAU)]:
                            r=predict(loadp(arm,seed,'fp32',split),rtau,gate); pos=y==1
                            k=int(((r==1)&(q==0)&pos).sum()); n=int(pos.sum()); detected=int(((r==1)&pos).sum())
                            FLIPS.append(dict(arm=arm,seed=seed,variant=v,split=split,system=system,
                                comparison=comparison,reference=refname,tau=tau,k=k,n_positive=n,
                                reference_detected=detected,rate=k/n if n else np.nan,
                                conditional_loss_rate=k/detected if detected else np.nan,
                                upper95_per_seed=zero_upper(k,n)))
    results=pd.DataFrame(ROWS); results.to_csv(ROOT/'tables/final_results.csv',index=False)
    pd.DataFrame(FLIPS).to_csv(ROOT/'tables/harmful_flips_per_seed.csv',index=False)
    pd.DataFrame(DELTAS).to_csv(ROOT/'tables/routing_paired_group_intervals.csv',index=False)
    print(results.groupby(['arm','analysis','system','split'])[['recall','fpr','stage2_rate']].agg(['mean','std']).round(4))
else: print('Final evaluation skipped.')


## 7. Seed-paired comparisons, error profiles and honest claim checks
Seed intervals describe variability over training seeds on these fixed data. They do not quantify new-source or population uncertainty. There are many exploratory comparisons: do not select the best-looking p-value or call an interval including zero proof of equivalence. R90 means matched **calibration target**, not equal observed test recall.


In [ ]:
if RUN_FINAL:
    paired=[]
    for analysis in ['calibration_R90','fixed_0.70','dev_selected']:
        for system in ['neural','hybrid','hybrid_v2']:
            r=results[(results.analysis==analysis)&(results.system==system)]
            for split in FINAL:
                for met in ['recall','fpr']:
                    a=r[(r.arm=='ce')&(r.split==split)][['seed','variant',met]]
                    b=r[(r.arm=='saqs')&(r.split==split)][['seed','variant',met]]
                    # Matched target uses a fixed variant; selected compares entire selection procedures.
                    variants=['procedure'] if analysis=='dev_selected' else ['fp32','qint8']
                    for variant in variants:
                        aa=a if variant=='procedure' else a[a.variant==variant]
                        bb=b if variant=='procedure' else b[b.variant==variant]
                        pairs=aa.merge(bb,on='seed',suffixes=('_ce','_saqs'))
                        if not len(pairs): continue
                        mean,lo,hi=seed_interval((pairs[f'{met}_saqs']-pairs[f'{met}_ce']).dropna()) if pairs[f'{met}_saqs'].notna().any() else (np.nan,np.nan,np.nan)
                        paired.append(dict(analysis=analysis,system=system,split=split,variant=variant,metric=met,
                            paired_seeds=len(pairs),saqs_minus_ce=mean,lo=lo,hi=hi,
                            scope='conditional on jointly accepted seeds' if analysis=='dev_selected' else 'all available paired seeds'))
    pd.DataFrame(paired).to_csv(ROOT/'tables/ce_saqs_paired_seed_intervals.csv',index=False)
    # Threshold-free discrimination: every arm/seed, same data. Independent of acceptance.
    discr=[]
    for item in PLAN.values():
        a,s=item['arm'],item['seed']
        for split in FINAL:
            y=DS[split].label.to_numpy()
            if len(np.unique(y))<2: continue
            for v in ['fp32']+VARIANTS:
                p=loadp(a,s,v,split)
                discr.append(dict(arm=a,seed=s,variant=v,split=split,auroc=roc_auc_score(y,p),auprc=average_precision_score(y,p)))
    pd.DataFrame(discr).to_csv(ROOT/'tables/discrimination.csv',index=False)
    # Profile counts for fixed qint8 hybrid v2 and all development-selected systems.
    profiles=[]; errors=[]
    for key,pr in PREDS.items():
        a,s,v,split,analysis,system=key
        if system!='hybrid_v2' or not (analysis=='dev_selected' or (analysis=='fixed_0.70' and v=='qint8')): continue
        d=DS[split].copy(); d['prediction']=pr
        for prof,g in d.groupby('profile'):
            profiles.append(dict(arm=a,seed=s,variant=v,split=split,analysis=analysis,profile=prof,**metric(g.label,g.prediction)))
        for _,r in d[d.label!=d.prediction].iterrows():
            errors.append(dict(arm=a,seed=s,variant=v,split=split,analysis=analysis,text=r.text,label=r.label,prediction=r.prediction,profile=r.profile))
    pd.DataFrame(profiles).to_csv(ROOT/'tables/error_profiles.csv',index=False)
    pd.DataFrame(errors).to_csv(ROOT/'tables/misclassified_texts.csv',index=False)
    claims=[]
    for arm in ARMS:
        items=[v for v in PLAN.values() if v['arm']==arm]; passed=sum(v['selected'] is not None for v in items)
        claims.append(f'{arm}: {passed}/{len(items)} seeds passed hybrid development budgets plus calibration recall guard. Report rejected seeds; no fallback is accepted.')
    claims += ['Previously observed final sets are replication evidence, not untouched confirmatory evaluation.',
      'Any original 6.2% hard-negative rate must be recalculated on independent, verified negatives; this notebook does not assume that number will reproduce.',
      'Zero flips describes observed comparative stability; inspect absolute recall and fixed-threshold flips separately.',
      'No clinical validation, clinical readiness or deployment-safety claim is established.',
      'Compression size is measured per ONNX file; device latency requires separate measurements on that device.',
      'New quantizer selection and additional recall guard are revision methods, not the original fixed seed-42/0.63 configuration.']
    (ROOT/'tables/claim_check.txt').write_text('\n'.join(claims)); print('\n'.join(claims))
    import matplotlib.pyplot as plt
    r=results[(results.analysis=='calibration_R90')&(results.variant=='qint8')&(results.system=='hybrid')]
    if len(r):
        fig,ax=plt.subplots(figsize=(7,4))
        for arm,g in r[r.split=='test'].groupby('arm'): ax.scatter(g.fpr,g.recall,label=arm,alpha=.7)
        ax.set(xlabel='Dataset-label FPR',ylabel='Dataset-label recall',title='Fixed INT8 backend; target chosen on calibration only')
        ax.legend(); fig.tight_layout(); fig.savefig(ROOT/'figures/recall_fpr.png',dpi=180); plt.show()
else: print('Analysis skipped until RUN_FINAL=True.')


## 8. Non-clinical annotation support (optional)
You do not need to hire clinicians to run this notebook. Two independent readers can assess whether a text expresses a first-person disclosure, contains only quotation/third-party discussion, or is unclear. This validates a linguistic labeling task, not medical risk. Preserve unclear/disagreed cases and disclose their counts. Source membership alone does not guarantee a text is negative. Independently adjudicate disagreements under a written guideline.

The unbiased random sample estimates labeling problems; a separate flagged sample supports error analysis. Do not remove only one model's alerts and then call the remaining full set a corrected benchmark. Compute verified-negative performance on the same adjudicated subset for every model, and describe how that subset was sampled.


In [ ]:
ann=DS['hn_final'].sample(n=min(150,len(DS['hn_final'])),random_state=20261110).copy()
ann['annotator1']=''; ann['annotator2']=''; ann['adjudicated']=''
p=ROOT/'data/hn_random_annotation_template.csv'
if not p.exists(): ann.to_csv(p,index=False)
(ROOT/'data/annotation_guideline.txt').write_text(
    'Independently label text: 0=no first-person suicidal/self-harm disclosure; '
    '1=first-person disclosure or intent; 2=unclear/context insufficient. '
    'Do not infer medical status. Distinguish quoted and third-party speech. '
    'Keep independent labels, discuss disagreements only afterwards, and record adjudicated labels. '
    'Record annotator qualifications, sampling, guideline, uncertainty, and disagreement counts. '
    'Use group_id to preserve conversations/template families. This is a non-clinical text-labeling task.')
completed=(Path(FRESH_DIR)/'hn_random_annotated.csv') if FRESH_DIR else None
if completed and completed.exists():
    a=pd.read_csv(completed); assert {'text','annotator1','annotator2','adjudicated'}<=set(a.columns)
    for c in ['annotator1','annotator2','adjudicated']: a[c]=pd.to_numeric(a[c],errors='coerce')
    assert a.text.map(text_hash).isin(DS['hn_final'].audit_hash).all()
    assert not a.text.map(text_hash).duplicated().any()
    valid=a.annotator1.isin([0,1,2])&a.annotator2.isin([0,1,2])
    print('Independent-label kappa:',cohen_kappa_score(a.loc[valid,'annotator1'],a.loc[valid,'annotator2']) if valid.any() else np.nan)
    print('Adjudicated counts (retain 2/unclear):',a.adjudicated.value_counts(dropna=False).to_dict())
    hashes=set(a.loc[a.adjudicated==0,'text'].map(text_hash)); keep=DS['hn_final'].audit_hash.isin(hashes).to_numpy()
    audited=[]
    if RUN_FINAL and keep.any():
        for key,pr in PREDS.items():
            arm,seed,variant,split,analysis,system=key
            if split=='hn_final': audited.append(dict(arm=arm,seed=seed,variant=variant,analysis=analysis,system=system,
                verified_negative_n=int(keep.sum()),alert_rate=float(pr[keep].mean()),scope='same adjudicated subset; sampled not full benchmark'))
        pd.DataFrame(audited).to_csv(ROOT/'tables/verified_negative_subset.csv',index=False)
print('Optional annotation template:',p)


## 9. Mobile handoff and results archive
Only a development-accepted SAQS seed can be exported as an accepted candidate. Choose the first passing seed in the predetermined seed order, never the best final result. No accepted seed means no accepted mobile export. Record device model, OS, runtime, threads, execution provider, tokenizer, warm-up, repetitions and thermal conditions. Time tokenization, rule routing, model execution and complete processing separately. A CPU Kaggle measurement is not a phone measurement.

On each target device, verify prediction parity against these per-message CPU references, then measure latency for neural-only and full hybrid on the same traffic inputs. Changed kernels may change decisions and require a separately documented validation protocol.


In [ ]:
if RUN_FINAL:
    eligible=[PLAN[tag('saqs',s)] for s in SEEDS if PLAN[tag('saqs',s)]['selected'] is not None]
    if eligible:
        item=eligible[0]; s=item['seed']; v=item['selected']['variant']; tau=item['selected']['tau']
        dest=ROOT/'mobile'; shutil.copy2(model_dir('saqs',s)/f'{v}.onnx',dest/'model.onnx')
        tok=AutoTokenizer.from_pretrained(str(model_dir('saqs',s))); tok.save_pretrained(dest)
        # Export gate implementation and constants as a portable Python reference.
        gatefile='import re, json, hashlib\nimport numpy as np\n'+inspect.getsource(normalize_s1)
        gatefile+='\nV2_CRISIS_RULES = '+repr(V2_CRISIS_RULES)+'\nV2_SAFE_RULES = '+repr(V2_SAFE_RULES)
        gatefile+='\nV2_NEGATORS = '+repr(V2_NEGATORS)+'\nV2_CONTRAST = re.compile('+repr(V2_CONTRAST.pattern)+')'
        gatefile+='\nV2_RISK_CUES = re.compile('+repr(V2_RISK_CUES.pattern)+')\n'
        gatefile+='_C=[(i,c,re.compile(p)) for i,c,p in V2_CRISIS_RULES]\n_S=[(i,c,re.compile(p)) for i,c,p in V2_SAFE_RULES]\n'
        gatefile+=inspect.getsource(_is_negated)+'\n'+inspect.getsource(stage1_v2)
        (dest/'gate_reference.py').write_text(gatefile)
        atomic_json(dest/'protocol.json',dict(arm='saqs',seed=s,variant=v,tau=tau,accepted_on='development only',
            protocol_hash=PROTOCOL_HASH,selection_hash=SELECTION_HASH,inference_batch=INFERENCE_BATCH,
            max_length=MAX_LENGTH,providers=['CPUExecutionProvider'],padding='max_length'))
        for split in ['probe_final','hn_final']+FRESH:
            d=DS[split]; p=loadp('saqs',s,v,split); gate=GATES['v2'][split]
            rec=d[['text','label','group_id']].copy(); rec['stage1']=gate; rec['cpu_probability']=p
            rec['hybrid_prediction']=predict(p,tau,gate); rec.to_csv(dest/f'{split}_parity.csv',index=False)
        print('Accepted mobile candidate:',s,v,tau)
    else: print('No accepted SAQS seed: mobile export intentionally omitted.')
# Packaging-only fix. No retraining, selection changes, or weight deletion.
from pathlib import Path
import json, zipfile, shutil, hashlib
ROOT = Path(globals().get('ROOT', '/kaggle/working/crisisq_nonclinical_v1_compat'))
if not (ROOT/'selection_frozen.json').exists():
    raise FileNotFoundError(f'Completed run not found at {ROOT}. Keep the original Kaggle session/output. Set ROOT to its saved directory before running this cell.')
# This exact archive was being produced when the logged run ran out of space.
# The source model directory and original input archive are preserved.
partial = ROOT.parent/(ROOT.name+'_development.zip')
if partial.exists():
    size=partial.stat().st_size
    partial.unlink()
    print(f'Removed interrupted generated archive ({size/1e9:.2f} GB); original files preserved.')

# Explicit allowlist: do not duplicate the many gigabytes of model weights.
files=[]
for subdir in ['tables','cache','figures','data']:
    folder=ROOT/subdir
    if folder.exists(): files.extend(p for p in folder.rglob('*') if p.is_file() and not p.is_symlink())
files.extend(p for p in ROOT.glob('*.json') if p.is_file() and not p.is_symlink())
files=sorted(set(files))
raw_size=sum(p.stat().st_size for p in files)
free=shutil.disk_usage(ROOT).free
print(f'Free disk: {free/1e9:.2f} GB; report/cache payload: {raw_size/1e6:.1f} MB')
if free < raw_size+128*1024**2:
    raise RuntimeError('Insufficient free space for the small results package. Preserve the run directory; do not delete weights or caches.')
phase='final' if (ROOT/'tables/final_results.csv').exists() else 'development'
target=ROOT.parent/(ROOT.name+'_'+phase+'_reports.zip')
tmp=target.with_suffix('.partial.zip')
with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for p in files: z.write(p,p.relative_to(ROOT))
    z.writestr('PACKAGE_README.txt',
        'Reports, probability caches, annotations, and frozen manifests only. '
        'Model weights remain in the original run/models directory. '
        'Keep that directory for final inference and mobile export. '
        'This packaging patch does not change training, selection, rules, budgets, or evaluation.\n')
    z.writestr('packaging_patch.json',json.dumps(dict(
        patch='allowlisted_reports_only_v1',omitted=['models','original_input','mobile'],
        original_protocol_sha256=hashlib.sha256((ROOT/'protocol_frozen.json').read_bytes()).hexdigest()),indent=2))
with zipfile.ZipFile(tmp) as z:
    bad=z.testzip()
    if bad: raise RuntimeError(f'ZIP integrity check failed: {bad}')
tmp.replace(target)
plan=json.loads((ROOT/'selection_frozen.json').read_text())
counts={}
for item in plan['models'].values():
    arm=item['arm']; counts.setdefault(arm,[0,0]); counts[arm][1]+=1
    counts[arm][0]+=int(item['selected'] is not None)
print('Development-selected seeds:',counts)
print('Download:',target, f'({target.stat().st_size/1e6:.1f} MB)')
print('Keep models/ and cache/. Final evaluation can resume in the SAME run directory.')
if globals().get('SMOKE_TEST',False): print('SMOKE TEST ONLY: not publication results.')


## Reading the result

* Original fixed configuration succeeds on genuinely independent data: evidence for that narrowly specified configuration. Report all predefined metrics, not just the best one.
* Only a new variant passes: evidence for a revised method. State explicitly what changed and that these are post-hoc experiments.
* No variant passes: the correct result is rejection under the stated budgets. Do not relax budgets after inspecting final results and claim confirmation.
* Hybrid improves recall but increases alert rate: report the tradeoff. If hybrid does not reduce HN alerts relative to the same neural model, do not attribute HN improvements to routing.
* Zero flips plus low absolute recall: stability of a detector that still misses positives; not demonstrated safety.
* Only previously observed data available: replication/exploratory evidence, with independent evaluation still outstanding.

### Technical references
Implementation was checked against [ONNX Runtime quantization documentation](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html) and [Hugging Face Trainer documentation](https://huggingface.co/docs/transformers/en/main_classes/trainer). Reduced-range and unsigned-weight experiments are backend alternatives, not a proven explanation for your earlier failures. Runtime versions and export parity are recorded in the experiment.
